# NB-04: Military Technology Score Construction

**The Peacekeepers' Arms Race — Stability-Instability Paradox**

Constructs four Military Technology Score (MTS) columns from SIPRI milex, SIPRI TIV imports, and CoW CINC components. Validates against CINC (1989–2016), visualises distributions and trajectories, then propagates MTS columns to all three RQ panels.

**Specs produced:** `mts_milex` (1946–2024), `mts_tiv` (1989–2024, NaN where no 5yr imports), `mts_pca` (1989–2016, 4-feat validation only), `mts_pca_3feat` (1989–2024, primary analytical spec).

## Section 0 — Setup

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA

from src.io_utils import load_checkpoint, save_checkpoint
from src.config import CLEAN_DIR, FIGURES_DIR, TABLES_DIR

plt.rcParams.update({
    "figure.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
sns.set_palette("tab10")

FIG_DIR = FIGURES_DIR / "nb04"
TBL_DIR = TABLES_DIR / "nb04"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TBL_DIR.mkdir(parents=True, exist_ok=True)

master = load_checkpoint(CLEAN_DIR / "master_panel.parquet")
print(f"Shape: {master.shape}")
print(f"\nColumns ({len(master.columns)}):")
for col in master.columns:
    print(f"  {col}")

# Row count must be 15168; column count grows as MTS specs are added across runs
assert master.shape[0] == 15168, f"Expected 15168 rows, got {master.shape[0]}"
assert master.shape[1] >= 56, f"Expected at least 56 columns, got {master.shape[1]}"
print("\nSection 0 complete.")

[checkpoint] loaded ← master_panel.parquet  (15,168 rows)
Shape: (15168, 61)

Columns (61):
  iso3
  year
  acd_n_conflicts
  acd_n_war
  acd_n_minor
  acd_n_interstate
  acd_n_intrastate
  acd_n_extrasystemic
  acd_n_internationalised
  part_n_conflicts
  part_n_interstate
  part_n_intrastate
  part_n_internationalised
  part_n_extrasystemic
  part_n_war
  part_n_minor
  part_n_extraterritorial
  brd_deaths_best
  brd_deaths_low
  brd_deaths_high
  ged_n_events
  ged_deaths_total
  sipri_milex_const2024_usd
  tiv_imports_total
  tiv_cat_air_defence_systems
  tiv_cat_aircraft
  tiv_cat_armoured_vehicles
  tiv_cat_artillery
  tiv_cat_engines
  tiv_cat_missiles
  tiv_cat_naval_weapons
  tiv_cat_other
  tiv_cat_satellites
  tiv_cat_sensors
  tiv_cat_ships
  wb_gdp_usd
  wb_gdp_per_capita
  wb_population
  vdem_v2x_polyarchy
  cow_cinc
  cow_milper
  cow_milex
  has_military
  log_milex
  log_tiv_imports
  log_gdp
  log_pop
  war_minor_ratio
  milex_pct_gdp
  extraterritorial_share
  tiv_i

## Section 1 — MTS Module Construction

`src/mts.py` is the source of truth — edit it directly, not via `%%writefile`.

Computes four MTS specs; applies the zero-import NaN fix to `mts_tiv`; adds `tiv_ever_imported` binary indicator.

In [2]:
# src/mts.py is the source of truth — edit it directly, not via %%writefile.
import importlib
import src.mts as _mts_module
importlib.reload(_mts_module)
from src.mts import compute_mts_milex, compute_mts_tiv, compute_mts_pca, compute_mts_pca_3feat

# ── Pre-compute 5yr rolling TIV sum (needed for zero-import mask and diagnostics) ─
_sorted = master.sort_values(["iso3", "year"])
master["tiv_imports_5yr_sum"] = (
    _sorted.groupby("iso3")["tiv_imports_total"]
    .transform(lambda x: x.rolling(5, min_periods=1).sum())
    .reindex(master.index)
)

# ── Compute all four MTS specifications ──────────────────────────────────────────
master["mts_milex"]    = compute_mts_milex(master)
master["mts_tiv"]      = compute_mts_tiv(master)
master["mts_pca"]      = compute_mts_pca(master)
master["mts_pca_3feat"] = compute_mts_pca_3feat(master)

# ── FIX 1 DIAGNOSIS — quantify the mass point at mts_tiv == 0.0 ─────────────────
zero_tiv  = (master["tiv_imports_5yr_sum"] == 0).sum()
total_tiv = master["mts_tiv"].notna().sum()
print(f"\nFIX 1 DIAGNOSIS")
print(f"Country-years with tiv_imports_5yr_sum == 0: {zero_tiv:,} of {total_tiv:,}")
print(master["mts_tiv"].describe().round(4))
print(f"USA mts_tiv 2020 (pre-fix): {master[(master['iso3']=='USA') & (master['year']==2020)]['mts_tiv'].values}")

# ── FIX 1 APPLY — NaN where no acquisition activity in the 5yr window ────────────
# Country-years with zero 5yr TIV sum are not being measured by this proxy —
# assigning them the MinMax floor contaminates continuous-variation assumptions in OLS/FE.
master["mts_tiv"] = np.where(
    master["tiv_imports_5yr_sum"] == 0,
    np.nan,
    master["mts_tiv"],
)

# ── SUPPLEMENTARY — binary acquisition indicator ──────────────────────────────────
master["tiv_ever_imported"] = (master["tiv_imports_5yr_sum"] > 0).astype(float)
master.loc[master["tiv_imports_5yr_sum"].isna(), "tiv_ever_imported"] = np.nan

# ── VERIFY Fix 1 ─────────────────────────────────────────────────────────────────
print(f"\nFIX 1 VERIFY")
n_zero_remaining = int((master["mts_tiv"] == 0).sum())
n_nonan = int(master["mts_tiv"].notna().sum())
print(f"mts_tiv non-NaN: {n_nonan:,}")
print(f"  Note: compute_mts_tiv already returns NaN for zero-5yr-sum rows;")
print(f"  the np.where guard above is a redundant safety check, not the primary filter.")
print(f"mts_tiv == 0 remaining: {n_zero_remaining} (MinMax floor of smallest positive value; ≤ 1 expected)")
usa_tiv_post = master[(master["iso3"] == "USA") & (master["year"] == 2020)]["mts_tiv"]
print(f"USA mts_tiv 2020: {usa_tiv_post.values[0]:.4f} (USA is a moderate importer; positive value expected)")

# ── FIX 2 VERIFY — coverage extension for mts_pca_3feat ────────────────────────
print(f"\nFIX 2 COVERAGE CHECK")
for spec in ["mts_pca", "mts_pca_3feat"]:
    sub = master[master[spec].notna()]
    print(f"{spec}: {sub['year'].min()}–{sub['year'].max()}, {sub['iso3'].nunique()} countries")

print()
for col in ["mts_milex", "mts_tiv", "mts_pca", "mts_pca_3feat"]:
    n = master[col].notna().sum()
    print(f"{col}: {n:,} non-NaN")

print("\nSection 1 complete.")

mts_pca PCA explained variance ratio (PC1): 0.6371
mts_pca_3feat PCA explained variance ratio (PC1): 0.6082

FIX 1 DIAGNOSIS
Country-years with tiv_imports_5yr_sum == 0: 5,397 of 9,771
count    9771.0000
mean        0.5029
std         0.2331
min         0.0000
25%         0.3236
50%         0.5131
75%         0.6982
max         1.0000
Name: mts_tiv, dtype: float64
USA mts_tiv 2020 (pre-fix): [0.80120137]

FIX 1 VERIFY
mts_tiv non-NaN: 9,771
  Note: compute_mts_tiv already returns NaN for zero-5yr-sum rows;
  the np.where guard above is a redundant safety check, not the primary filter.
mts_tiv == 0 remaining: 1 (MinMax floor of smallest positive value; ≤ 1 expected)
USA mts_tiv 2020: 0.8012 (USA is a moderate importer; positive value expected)

FIX 2 COVERAGE CHECK
mts_pca: 1960–2016, 155 countries
mts_pca_3feat: 1960–2024, 157 countries

mts_milex: 8,212 non-NaN
mts_tiv: 9,771 non-NaN
mts_pca: 6,250 non-NaN
mts_pca_3feat: 7,550 non-NaN

Section 1 complete.


### MTS PCA specification note

Two PCA variants are produced:

- **`mts_pca` (4-feat):** includes `log_milper` from CoW CINC. Covers 1989–2016. Used for construct validation (ρ ≈ 0.91 with CINC). **Not used as primary analytical spec** due to 2016 truncation.
- **`mts_pca_3feat`:** drops `log_milper`. Covers 1989–2024. Used as primary MTS in NB05–NB07 regressions and clustering. ρ with CINC should be > 0.80.

**Primary spec for NB05 onwards: `mts_pca_3feat`.**

## Section 2 — CINC Validation (Spearman)

Two complementary tests: (1) country-means Spearman asks whether high-capability countries rank consistently high in both MTS and CINC; (2) panel-level Spearman asks whether MTS tracks within-country capability growth over time. Both are needed for construct validity.

In [3]:
cinc_sub = master.loc[
    (master["year"] >= 1989) & (master["year"] <= 2016) & master["cow_cinc"].notna()
].copy()

country_means = (
    cinc_sub
    .groupby("iso3")[["mts_milex", "mts_tiv", "mts_pca", "mts_pca_3feat", "cow_cinc"]]
    .mean()
)

# ── Country-means Spearman (cross-country ranking) ────────────────────────────
country_rows = []
for spec in ["mts_milex", "mts_tiv", "mts_pca", "mts_pca_3feat"]:
    pair = country_means[[spec, "cow_cinc"]].dropna()
    rho, pval = spearmanr(pair[spec], pair["cow_cinc"])
    if rho > 0.7:
        interp = "Strong agreement with CINC"
    elif rho >= 0.4:
        interp = "Moderate agreement \u2014 MTS captures additional variance"
    else:
        interp = "Weak agreement \u2014 flag for writeup"
    country_rows.append({
        "MTS Spec":       spec,
        "Spearman rho":   round(rho,  4),
        "p-value":        round(pval, 6),
        "n_countries":    len(pair),
        "Interpretation": interp,
    })

cinc_val = pd.DataFrame(country_rows)
print("=== Country-means Spearman (cross-country ranking) ===")
print(cinc_val.to_string(index=False))

# ── Panel-level Spearman (within-country variation) ────────────────────────────
panel_rows = []
for spec in ["mts_milex", "mts_tiv", "mts_pca", "mts_pca_3feat"]:
    sub = cinc_sub[cinc_sub[spec].notna()][[spec, "cow_cinc"]].dropna()
    rho, pval = spearmanr(sub[spec], sub["cow_cinc"])
    panel_rows.append({
        "MTS Spec":        spec,
        "rho_panel":       round(rho,  4),
        "p-value":         round(pval, 6),
        "n_country_years": len(sub),
    })

panel_val = pd.DataFrame(panel_rows)
print("\n=== Panel-level Spearman (country-year, tests within-country variation) ===")
print(panel_val.to_string(index=False))

cinc_val.to_csv(TBL_DIR / "cinc_validation.csv",       index=False)
panel_val.to_csv(TBL_DIR / "panel_cinc_validation.csv", index=False)
print(f"\nSaved \u2192 {TBL_DIR}")
print("\nSection 2 complete.")

=== Country-means Spearman (cross-country ranking) ===
     MTS Spec  Spearman rho  p-value  n_countries             Interpretation
    mts_milex        0.8469      0.0          159 Strong agreement with CINC
      mts_tiv        0.8292      0.0          175 Strong agreement with CINC
      mts_pca        0.9113      0.0          155 Strong agreement with CINC
mts_pca_3feat        0.8417      0.0          155 Strong agreement with CINC

=== Panel-level Spearman (country-year, tests within-country variation) ===
     MTS Spec  rho_panel  p-value  n_country_years
    mts_milex     0.8483      0.0             3926
      mts_tiv     0.7825      0.0             3934
      mts_pca     0.8960      0.0             3830
mts_pca_3feat     0.8118      0.0             3830



Saved → D:\post graduate\assignments and projects\sem 3\BDA\PeaceMakers' Arms Race\version 2\tables\nb04

Section 2 complete.


### Validation interpretation

**Country-means Spearman** asks: do high-capability countries rank consistently high in both MTS and CINC? This tests cross-sectional validity.

**Panel Spearman** asks: when a country’s capability grows, does MTS track that growth? This tests temporal (within-country) validity.

Both tests are needed for construct validity. A spec can rank countries well (high country-means ρ) while being insensitive to year-over-year change (low panel ρ), or vice versa.

## Section 3 — Visualizations

Four figures. Fig 3 uses a 3-panel layout (one per primary spec) to avoid the silent 2016 cutoff that plagued a single-spec trajectory chart.

In [4]:
# ── Fig 1: MTS Distribution (violin, all four specs) ──────────────────────────
scope   = master.loc[master["year"] >= 1989, ["mts_milex", "mts_tiv", "mts_pca", "mts_pca_3feat"]]
melt_df = scope.melt(var_name="MTS Specification", value_name="Score").dropna()

fig, ax = plt.subplots(figsize=(12, 5))
sns.violinplot(data=melt_df, x="MTS Specification", y="Score", ax=ax, inner="box")
ax.set_title("Distribution of Four MTS Specifications (1989\u20132024)")
ax.set_xlabel("MTS Specification")
ax.set_ylabel("Score (0\u20131)")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig1_mts_distributions.png", dpi=300)
plt.close(fig)
print("Fig 1 saved.")

# ── Fig 2: CINC Scatter (3-panel, mts_milex / mts_tiv / mts_pca) ──────────────
specs       = ["mts_milex", "mts_tiv", "mts_pca"]
spec_titles = ["mts_milex vs CINC", "mts_tiv vs CINC", "mts_pca vs CINC"]
rho_map     = {row["MTS Spec"]: row["Spearman rho"] for _, row in cinc_val.iterrows()}
top15       = country_means.nlargest(15, "cow_cinc").index

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
for ax, spec, title in zip(axes, specs, spec_titles):
    pair = country_means[[spec, "cow_cinc"]].dropna()
    ax.scatter(pair[spec], pair["cow_cinc"], alpha=0.5, s=20, color="steelblue")
    for iso3 in top15:
        if iso3 in pair.index:
            ax.text(
                pair.loc[iso3, spec], pair.loc[iso3, "cow_cinc"],
                iso3, fontsize=7, ha="left", va="bottom",
            )
    coeffs = np.polyfit(pair[spec], pair["cow_cinc"], 1)
    xline  = np.linspace(pair[spec].min(), pair[spec].max(), 100)
    ax.plot(xline, np.polyval(coeffs, xline), color="firebrick", lw=1.5)
    rho = rho_map.get(spec, float("nan"))
    ax.set_title(f"{title}\n(\u03c1 = {rho:.3f})")
    ax.set_xlabel(spec)
axes[0].set_ylabel("CINC (1989\u20132016 mean)")
fig.suptitle("MTS vs CINC Validation \u2014 Country-Level Means (1989\u20132016)", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2_cinc_scatter.png", dpi=300, bbox_inches="tight")
plt.close(fig)
print("Fig 2 saved.")

# ── Fig 3: Headline Country Trajectories (3-panel, one per primary spec) ──────
# 3-panel layout avoids the silent 2016 cutoff from a single mts_pca trajectory.
highlight_countries = ["USA", "RUS", "CHN", "IND", "GBR", "SAU"]
traj_specs  = ["mts_milex", "mts_tiv", "mts_pca_3feat"]
traj_titles = [
    "MTS: MILEX (1946\u20132024)",
    "MTS: TIV imports (1989\u20132024)",
    "MTS: PCA 3-feat (1989\u20132024)",
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, spec, title in zip(axes, traj_specs, traj_titles):
    for iso3 in highlight_countries:
        sub = master[(master["iso3"] == iso3) & master[spec].notna()].sort_values("year")
        if not sub.empty:
            ax.plot(sub["year"], sub[spec], label=iso3, linewidth=1.5)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Year")
    ax.set_ylabel("MTS (0\u20131 scaled)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig3_mts_trajectories.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Fig 3 saved.")

# ── Fig 4: Correlation Matrix (all four specs + CINC) ─────────────────────────
corr_cols = ["mts_milex", "mts_tiv", "mts_pca", "mts_pca_3feat", "cow_cinc"]
corr_sub  = master.loc[
    (master["year"] >= 1989) & (master["year"] <= 2016), corr_cols
].dropna()
corr_mat  = corr_sub.corr(method="pearson")

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    corr_mat, annot=True, fmt=".2f", cmap="coolwarm",
    vmin=-1, vmax=1, square=True, ax=ax, linewidths=0.5,
)
ax.set_title("Correlation Matrix: MTS Specifications and CINC")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig4_mts_correlation_matrix.png", dpi=300)
plt.close(fig)
print("Fig 4 saved.")

print("\nSection 3 complete.")

Fig 1 saved.


Fig 2 saved.


Fig 3 saved.


Fig 4 saved.

Section 3 complete.


## Section 4 — Propagate MTS to RQ Panels

In [5]:
mts_cols   = ["mts_milex", "mts_tiv", "mts_pca", "mts_pca_3feat", "tiv_ever_imported"]
mts_lookup = master[["iso3", "year"] + mts_cols]

# ── RQ1 panel ──────────────────────────────────────────────────────────────────
rq1 = load_checkpoint(CLEAN_DIR / "rq1_panel.parquet")
rq1_before = rq1.shape[0]
rq1 = rq1.drop(columns=[c for c in mts_cols if c in rq1.columns])
rq1 = rq1.merge(mts_lookup, on=["iso3", "year"], how="left")
assert rq1.shape[0] == rq1_before, f"Fan-out in rq1_panel: {rq1_before} \u2192 {rq1.shape[0]}"
save_checkpoint(rq1, CLEAN_DIR / "rq1_panel.parquet")
print(f"rq1_panel:            {rq1_before:,} \u2192 {rq1.shape[0]:,} rows")

# ── RQ2 treatment annual ────────────────────────────────────────────────────────
rq2 = load_checkpoint(CLEAN_DIR / "rq2_treatment_annual.parquet")
rq2_before = rq2.shape[0]
rq2 = rq2.drop(columns=[c for c in mts_cols if c in rq2.columns])
rq2 = rq2.merge(mts_lookup, on=["iso3", "year"], how="left")
assert rq2.shape[0] == rq2_before, f"Fan-out in rq2: {rq2_before} \u2192 {rq2.shape[0]}"
save_checkpoint(rq2, CLEAN_DIR / "rq2_treatment_annual.parquet")
print(f"rq2_treatment_annual: {rq2_before:,} \u2192 {rq2.shape[0]:,} rows")

# ── RQ3 cross-section (iso3 means, 1989–2024 filter) — FIX 3 ───────────────────
# Using 1989–2024 year range consistent with the rest of rq3_cross_section.
rq3 = load_checkpoint(CLEAN_DIR / "rq3_cross_section.parquet")
rq3_before = rq3.shape[0]
rq3_mts_cols = ["mts_milex", "mts_tiv", "mts_pca", "mts_pca_3feat"]
rq3 = rq3.drop(columns=[c for c in rq3_mts_cols if c in rq3.columns])
mts_iso3_means = (
    master[master["year"].between(1989, 2024)]
    .groupby("iso3")[rq3_mts_cols]
    .mean()
    .reset_index()
)
rq3 = rq3.merge(mts_iso3_means, on="iso3", how="left")
assert rq3.shape[0] == rq3_before, f"Fan-out in rq3: {rq3_before} \u2192 {rq3.shape[0]}"
save_checkpoint(rq3, CLEAN_DIR / "rq3_cross_section.parquet")
print(f"rq3_cross_section:    {rq3_before:,} \u2192 {rq3.shape[0]:,} rows")
print(f"rq3 MTS coverage: {rq3['mts_milex'].notna().sum()} of {len(rq3)} countries")

print("\nSection 4 complete.")

[checkpoint] loaded ← rq1_panel.parquet  (6,912 rows)
[checkpoint] saved → rq1_panel.parquet  (6,912 rows)
rq1_panel:            6,912 → 6,912 rows
[checkpoint] loaded ← rq2_treatment_annual.parquet  (6,912 rows)
[checkpoint] saved → rq2_treatment_annual.parquet  (6,912 rows)
rq2_treatment_annual: 6,912 → 6,912 rows
[checkpoint] loaded ← rq3_cross_section.parquet  (192 rows)
[checkpoint] saved → rq3_cross_section.parquet  (192 rows)
rq3_cross_section:    192 → 192 rows
rq3 MTS coverage: 161 of 192 countries

Section 4 complete.


## Section 5 — Sanity Checks

In [6]:
results = []

def chk(label, expr):
    status = "PASS" if expr else "FAIL"
    results.append((label, status))
    print(f"[{status}] {label}")

# [1] Row count preserved
chk("[1] master.shape[0] == 15168", master.shape[0] == 15168)

# [2] All MTS columns and binary indicator present
chk(
    "[2] mts_milex/tiv/pca/pca_3feat/tiv_ever_imported in master",
    all(c in master.columns for c in ["mts_milex", "mts_tiv", "mts_pca", "mts_pca_3feat", "tiv_ever_imported"]),
)

# [3] mts_milex in [0, 1]
milex_vals = master["mts_milex"].dropna()
chk("[3] mts_milex in [0, 1]", bool(milex_vals.between(0, 1).all()))

# [4] mts_tiv mass point eliminated (at most 1 zero: the MinMax min of positive values)
chk("[4] mts_tiv mass point gone (<=1 zero)", (master["mts_tiv"] == 0).sum() <= 1)

# [5] mts_pca in [0, 1]
pca_vals = master["mts_pca"].dropna()
chk("[5] mts_pca in [0, 1]", len(pca_vals) == 0 or bool(pca_vals.between(0, 1).all()))

# [6] mts_pca_3feat extends to 2024 (Fix 2)
max_yr_3feat = int(master[master["mts_pca_3feat"].notna()]["year"].max())
chk(f"[6] mts_pca_3feat extends to 2024 (max year = {max_yr_3feat})", max_yr_3feat == 2024)

# [7] USA mts_pca_3feat rank in top 5 in most recent available year
yr_target = int(master[master["mts_pca_3feat"].notna()]["year"].max())
yr_df     = master[(master["year"] == yr_target) & master["mts_pca_3feat"].notna()][["iso3", "mts_pca_3feat"]]
ranked    = yr_df.sort_values("mts_pca_3feat", ascending=False).reset_index(drop=True)
usa_rank  = ranked[ranked["iso3"] == "USA"].index[0] + 1 if "USA" in ranked["iso3"].values else 999
chk(f"[7] USA mts_pca_3feat rank ({yr_target}) in top 5", usa_rank <= 5)
print(f"    USA {yr_target} rank: {usa_rank}")

# [8] rq1_panel has all four MTS columns
chk(
    "[8] rq1_panel has mts_milex, mts_tiv, mts_pca, mts_pca_3feat",
    all(c in rq1.columns for c in ["mts_milex", "mts_tiv", "mts_pca", "mts_pca_3feat"]),
)

# [9] No fan-out in rq1_panel
chk("[9] rq1_panel.shape[0] unchanged after merge", rq1.shape[0] == rq1_before)

# [10] USA mts_tiv 2020 is positive (USA has SIPRI TIV import records)
usa_tiv_2020 = master[(master["iso3"] == "USA") & (master["year"] == 2020)]["mts_tiv"]
chk("[10] USA mts_tiv 2020 is positive (SIPRI shows US imports)", usa_tiv_2020.notna().all() and float(usa_tiv_2020.iloc[0]) > 0)

n_pass = sum(1 for _, r in results if r == "PASS")
n_fail = sum(1 for _, r in results if r == "FAIL")
print(f"\n{n_pass}/10 checks passed, {n_fail} failed")

[PASS] [1] master.shape[0] == 15168
[PASS] [2] mts_milex/tiv/pca/pca_3feat/tiv_ever_imported in master
[PASS] [3] mts_milex in [0, 1]
[PASS] [4] mts_tiv mass point gone (<=1 zero)
[PASS] [5] mts_pca in [0, 1]
[PASS] [6] mts_pca_3feat extends to 2024 (max year = 2024)
[PASS] [7] USA mts_pca_3feat rank (2024) in top 5
    USA 2024 rank: 2
[PASS] [8] rq1_panel has mts_milex, mts_tiv, mts_pca, mts_pca_3feat
[PASS] [9] rq1_panel.shape[0] unchanged after merge
[PASS] [10] USA mts_tiv 2020 is positive (SIPRI shows US imports)

10/10 checks passed, 0 failed


## Final Validation and Save

In [7]:
# ── Strict assertions ───────────────────────────────────────────────────────
print("=== Strict Assertions ===")

assert (master["mts_tiv"] == 0).sum() <= 1, \
    "mts_tiv mass point not eliminated \u2014 more than 1 zero value remains"

assert master[master["mts_pca_3feat"].notna()]["year"].max() == 2024, \
    "mts_pca_3feat does not extend to 2024 \u2014 check WB GDP or SIPRI milex coverage"

assert master[master["mts_pca"].notna()]["year"].max() == 2016, \
    "mts_pca year range is not 2016 \u2014 cow_milper data range changed unexpectedly"

_rq3_check = pd.read_parquet(str(CLEAN_DIR / "rq3_cross_section.parquet"))
assert all(c in _rq3_check.columns for c in ["mts_milex", "mts_tiv", "mts_pca_3feat"]), \
    "rq3_cross_section missing MTS columns \u2014 re-run Section 4"

_usa_tiv = master[(master["iso3"] == "USA") & (master["year"] == 2020)]["mts_tiv"]
assert _usa_tiv.notna().all() and float(_usa_tiv.iloc[0]) > 0, \
    "USA mts_tiv 2020 should be positive \u2014 USA has SIPRI TIV import records"

print("All assertions passed.")

# ── Save updated master_panel ────────────────────────────────────────────────
save_checkpoint(master, CLEAN_DIR / "master_panel.parquet")
print(f"\nmaster_panel saved: {master.shape}")
new_cols = ["mts_milex", "mts_tiv", "mts_pca", "mts_pca_3feat", "tiv_imports_5yr_sum", "tiv_ever_imported"]
print(f"New columns added: {', '.join(new_cols)}")
print()
print(master[["mts_milex", "mts_tiv", "mts_pca", "mts_pca_3feat"]]
      .describe().T[["count", "mean", "min", "max"]]
      .round(3)
      .to_string())

n_milex = int(master["mts_milex"].notna().sum())
n_tiv   = int(master["mts_tiv"].notna().sum())
n_pca   = int(master["mts_pca"].notna().sum())
n_3feat = int(master["mts_pca_3feat"].notna().sum())

print(f"""
=== HEADLINE SPEC FOR NB05 ===
Primary:    mts_pca_3feat  (1989\u20132024, \u03c1_CINC > 0.80, {n_3feat:,} non-NaN)
Robustness: mts_milex      (1946\u20132024, \u03c1_CINC ~ 0.85, {n_milex:,} non-NaN)
Robustness: mts_tiv        (1989\u20132024, NaN where 5yr_sum=0,  {n_tiv:,} non-NaN)
Validation: mts_pca        (1989\u20132016, \u03c1_CINC ~ 0.91, not in RQ1\u2013RQ3, {n_pca:,} non-NaN)

Figures saved to figures/nb04/
Tables  saved to tables/nb04/

=== NB04 FIXES COMPLETE \u2014 NB05 can begin ===""")

=== Strict Assertions ===
All assertions passed.


[checkpoint] saved → master_panel.parquet  (15,168 rows)

master_panel saved: (15168, 61)
New columns added: mts_milex, mts_tiv, mts_pca, mts_pca_3feat, tiv_imports_5yr_sum, tiv_ever_imported

                count   mean  min  max
mts_milex      8212.0  0.480  0.0  1.0
mts_tiv        9771.0  0.503  0.0  1.0
mts_pca        6250.0  0.411  0.0  1.0
mts_pca_3feat  7550.0  0.249  0.0  1.0

=== HEADLINE SPEC FOR NB05 ===
Primary:    mts_pca_3feat  (1989–2024, ρ_CINC > 0.80, 7,550 non-NaN)
Robustness: mts_milex      (1946–2024, ρ_CINC ~ 0.85, 8,212 non-NaN)
Robustness: mts_tiv        (1989–2024, NaN where 5yr_sum=0,  9,771 non-NaN)
Validation: mts_pca        (1989–2016, ρ_CINC ~ 0.91, not in RQ1–RQ3, 6,250 non-NaN)

Figures saved to figures/nb04/
Tables  saved to tables/nb04/

=== NB04 FIXES COMPLETE — NB05 can begin ===
